In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تمريرات المُحوِّل بالذكاء الاصطناعي
تمريرات المُحوِّل المدعومة بالذكاء الاصطناعي هي تمريرات تعمل كبديل مباشر للتمريرات "التقليدية" في Qiskit لبعض مهام التحويل. وهي في الغالب تُنتج نتائج أفضل من الخوارزميات الاستدلالية الموجودة (كعمق أقل وعدد أقل من بوابات CNOT)، كما أنها أسرع بكثير من خوارزميات التحسين كمحللات الإشباع البولياني. يمكن تشغيل تمريرات المُحوِّل بالذكاء الاصطناعي في بيئتك المحلية أو على السحابة باستخدام Qiskit Transpiler Service إذا كنت ضمن خطة IBM Quantum&reg; Premium أو Flex أو On-Prem (عبر IBM Quantum Platform API).

> **Note:** تمريرات المُحوِّل المدعومة بالذكاء الاصطناعي في مرحلة الإصدار التجريبي (beta) وقد تتغير.
>     إذا كان لديك ملاحظات أو تريد التواصل مع فريق التطوير، استخدم [قناة Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) هذه.

التمريرات المتاحة حاليًا هي:

**تمريرات التوجيه**
 - `AIRouting`: اختيار التخطيط وتوجيه الدائرة

**تمريرات تركيب الدوائر**
 - `AICliffordSynthesis`: تركيب دوائر Clifford
 - `AILinearFunctionSynthesis`: تركيب دوائر الدوال الخطية
 - `AIPermutationSynthesis`: تركيب دوائر التبديل
 - `AIPauliNetworkSynthesis`: تركيب دوائر شبكة Pauli

لاستخدام تمريرات المُحوِّل بالذكاء الاصطناعي، [ثبِّت أولًا حزمة `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). تفضل بزيارة [توثيق API الخاص بـ qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) للاطلاع على مزيد من المعلومات حول الخيارات المختلفة المتاحة.

## تشغيل تمريرات المُحوِّل بالذكاء الاصطناعي محليًا أو على السحابة
إذا أردت استخدام تمريرات المُحوِّل المدعومة بالذكاء الاصطناعي في بيئتك المحلية مجانًا، ثبِّت `qiskit-ibm-transpiler` مع بعض التبعيات الإضافية كما يلي:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

بدون هذه التبعيات الإضافية، تعمل تمريرات المُحوِّل المدعومة بالذكاء الاصطناعي على السحابة عبر Qiskit Transpiler Service (متاح فقط لمستخدمي خطة IBM Quantum Premium أو Flex أو On-Prem (عبر IBM Quantum Platform API)). بعد تثبيت التبعيات الإضافية، يصبح الوضع الافتراضي لتشغيل التمريرات هو استخدام جهازك المحلي.

## تمريرة التوجيه بالذكاء الاصطناعي
تعمل تمريرة `AIRouting` كمرحلة تخطيط ومرحلة توجيه في آنٍ واحد. يمكن استخدامها داخل `PassManager` كما يلي:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

هنا، يحدد `backend` خريطة الاقتران المستخدمة للتوجيه، ويحدد `optimization_level` (1 أو 2 أو 3) مقدار الجهد الحسابي المبذول في العملية (القيم الأعلى عادةً تُعطي نتائج أفضل لكنها تستغرق وقتًا أطول)، ويحدد `layout_mode` كيفية التعامل مع اختيار التخطيط.
يتضمن `layout_mode` الخيارات التالية:

- `keep`: يحترم هذا الوضع التخطيط الذي حددته تمريرات المُحوِّل السابقة (أو يستخدم التخطيط البسيط إذا لم يُحدَّد). يُستخدم عادةً فقط عندما يجب تشغيل الدائرة على Qubits محددة في الجهاز. وغالبًا ما يُنتج نتائج أضعف لأنه يملك مساحة أقل للتحسين.
- `improve`: يستخدم هذا الوضع التخطيط الذي حددته تمريرات المُحوِّل السابقة كنقطة بداية. مفيد عندما يكون لديك تخمين أولي جيد للتخطيط؛ مثلًا للدوائر المبنية بطريقة تتبع تقريبًا خريطة اقتران الجهاز. مفيد أيضًا إذا أردت تجربة تمريرات تخطيط محددة أخرى مع تمريرة `AIRouting`.
- `optimize`: هذا هو الوضع الافتراضي. يعمل بشكل أفضل مع الدوائر العامة التي قد لا تكون لديك فيها تخمينات جيدة للتخطيط. يتجاهل هذا الوضع اختيارات التخطيط السابقة.
- `local_mode`: يحدد هذا الخيار مكان تشغيل تمريرة `AIRouting`. إذا كانت `False`، تعمل `AIRouting` عن بُعد عبر Qiskit Transpiler Service. إذا كانت `True`، تحاول الحزمة تشغيل التمريرة في بيئتك المحلية مع الرجوع إلى وضع السحابة إذا لم تُوجد التبعيات اللازمة.

## تمريرات تركيب الدوائر بالذكاء الاصطناعي
تتيح لك تمريرات تركيب الدوائر بالذكاء الاصطناعي تحسين أجزاء من أنواع مختلفة من الدوائر ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford)، [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction)، [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation)، Pauli Network) عن طريق إعادة تركيبها. الطريقة الشائعة لاستخدام تمريرة التركيب هي كما يلي:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

يحترم التركيب خريطة اقتران الجهاز: يمكن تشغيله بأمان بعد تمريرات التوجيه الأخرى دون الإخلال بالدائرة، بحيث تظل الدائرة الكلية ملتزمة بقيود الجهاز. افتراضيًا، لن يستبدل التركيب الدائرة الفرعية الأصلية إلا إذا أدت الدائرة الفرعية المُركَّبة إلى تحسين الأصلية (يتحقق حاليًا من عدد بوابات CNOT فقط)، لكن يمكنك إجبار الاستبدال دائمًا بضبط `replace_only_if_better=False`.

تمريرات التركيب المتاحة من `qiskit_ibm_transpiler.ai.synthesis` هي:

- *AICliffordSynthesis*: تركيب لدوائر [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (كتل من بوابات `H` و`S` و`CX`). حاليًا حتى تسعة Qubits.
- *AILinearFunctionSynthesis*: تركيب لدوائر [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (كتل من بوابات `CX` و`SWAP`). حاليًا حتى تسعة Qubits.
- *AIPermutationSynthesis*: تركيب لدوائر [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (كتل من بوابات `SWAP`). متاح حاليًا لكتل 65 و33 و27 Qubit.
- *AIPauliNetworkSynthesis*: تركيب لدوائر شبكة Pauli (كتل من بوابات `H` و`S` و`SX` و`CX` و`RX` و`RY` و`RZ`). حاليًا حتى ستة Qubits.

نتوقع زيادة حجم الكتل المدعومة تدريجيًا.

تستخدم جميع التمريرات مجموعة خيوط (thread pool) لإرسال عدة طلبات بالتوازي. افتراضيًا، الحد الأقصى لعدد الخيوط هو عدد الأنوية زائد أربعة (القيم الافتراضية لكائن Python `ThreadPoolExecutor`). لكن يمكنك تحديد قيمتك الخاصة باستخدام معامل `max_threads` عند إنشاء التمريرة. على سبيل المثال، السطر التالي ينشئ تمريرة `AILinearFunctionSynthesis` ويسمح لها باستخدام ما يصل إلى 20 خيطًا كحد أقصى.